In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [15]:
df = pd.read_csv('cleaned_data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10326 entries, 0 to 10325
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            10326 non-null  int64  
 1   SeniorCitizen     10326 non-null  int64  
 2   Partner           10326 non-null  int64  
 3   Dependents        10326 non-null  int64  
 4   PhoneService      10326 non-null  int64  
 5   MultipleLines     10326 non-null  int64  
 6   InternetService   10326 non-null  int64  
 7   OnlineSecurity    10326 non-null  int64  
 8   OnlineBackup      10326 non-null  int64  
 9   DeviceProtection  10326 non-null  int64  
 10  TechSupport       10326 non-null  int64  
 11  StreamingTV       10326 non-null  int64  
 12  StreamingMovies   10326 non-null  int64  
 13  Contract          10326 non-null  int64  
 14  PaperlessBilling  10326 non-null  int64  
 15  PaymentMethod     10326 non-null  int64  
 16  MonthlyCharges    10326 non-null  float64
 17  Tota

In [3]:
X = df.drop(columns = ['Unnamed: 0', 'Churn'])
y = df['Churn']

In [4]:
#Lets divide dataset into training and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_fscore_support
from sklearn.metrics import accuracy_score
#Random Forest Classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

rf = RandomForestClassifier(random_state=42)
param_dist = {
    'n_estimators': randint(100, 400),
    'max_depth': [5, 10, 15, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [None, 'balanced']
}
rand_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=60,     # 60 random tries (vs 648 in Grid)
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=2,
    random_state=42
)
rand_rf.fit(X_train, y_train)
best_rf = rand_rf.best_estimator_
y_pred = best_rf.predict(X_test)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


In [8]:
y_pred = best_rf.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print("Train Accuracy:", accuracy)
y_pred = best_rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

Train Accuracy: 0.926997578692494
Test Accuracy: 0.834946757018393


In [16]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# def evaluate_model(name, model, X_train, y_train, X_test, y_test):
#     # Predictions
#     y_train_pred = model.predict(X_train)
#     y_test_pred = model.predict(X_test)

#     # Probabilities (for ROC-AUC)
#     if hasattr(model, "predict_proba"):
#         y_test_prob = model.predict_proba(X_test)[:, 1]
#     else:
#         # For SVM with probability=False
#         try:
#             y_test_prob = model.decision_function(X_test)
#         except:
#             y_test_prob = None

#     # Metrics
#     result = {
#         "Model": name,
#         "Train Accuracy": accuracy_score(y_train, y_train_pred),
#         "Test Accuracy": accuracy_score(y_test, y_test_pred),
#         "Train F1": f1_score(y_train, y_train_pred),
#         "Test F1": f1_score(y_test, y_test_pred),
#         "Train Recall": recall_score(y_train, y_train_pred),
#         "Test Recall": recall_score(y_test, y_test_pred),
#         "ROC-AUC": roc_auc_score(y_test, y_test_prob) if y_test_prob is not None else None
#     }
#     return result

In [17]:
# results = []
# # results.append(evaluate_model("Logistic Regression", best_model, X_train, y_train, X_test, y_test))
# # results.append(evaluate_model("Decision Tree", best_dt, X_train, y_train, X_test, y_test))
# results.append(evaluate_model("Random Forest", best_rf, X_train, y_train, X_test, y_test))
# # results.append(evaluate_model("SVM", best_svc, X_train, y_train, X_test, y_test))
# # results.append(evaluate_model("XGBoost", best_pipeline, X_train, y_train, X_test, y_test))   # pipeline handles its own SMOTE

# import pandas as pd
# df_results = pd.DataFrame(results)
# df_results

In [18]:
# df_results.sort_values(by=["Test F1", "Test Recall"], ascending=False)
# df_results

In [19]:
import pickle as pkl
pkl.dump(best_rf, open('model.pkl', 'wb'))